# Project 5 — Semantic Correspondence (AML 2025/2026)

**Team:** Johnprice Osagie (s344613) · Mario Lapadula (s338300) · Giorgia Pugliese (s344683) · Riccardo Bellanca (s346229)

---

### ⚡ Setup rapido
1. Vai su **Runtime → Change runtime type → T4 GPU**
2. Esegui le celle in ordine
3. Il training completo (20 epoch) dura ~90 minuti su T4

In [ ]:
# ── Cella 1: Clona il repo ──────────────────────────────────────────────────
import os

REPO_URL    = "https://github.com/Johnnyprice29/Project_AML.git"
BRANCH      = "osagie5"
PROJECT_DIR = "/content/Project_AML"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git checkout {BRANCH}
!git pull
print("✅ Repo pronto")

In [ ]:
# ── Cella 2: Verifica GPU ───────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ Nessuna GPU! Vai su Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Cella 3: Installa dipendenze ────────────────────────────────────────────
!pip install -r requirements.txt -q
print("✅ Dipendenze installate")

In [ ]:
# ── Cella 4: Dataset SPair-71k ──────────────────────────────────────────────
# OPZIONE A: scarica dal server Postech (2-5 sec su Colab)
!python dataloaders/download_spair.py --root ./data

# OPZIONE B (più veloce se hai il tar su Google Drive):
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/SPair-71k.tar.gz" /content/
# !mkdir -p ./data && tar -xzf /content/SPair-71k.tar.gz -C ./data/

print("✅ Dataset pronto")

In [ ]:
# ── Cella 5: Training Compatto (2.5 Ore) ───────────────────────────────────
# Tutte le estensioni abilitate: LoRA + Curriculum + Adaptive Window
!python train.py \
    --dataset_root ./data/SPair-71k \
    --backbone dinov2_vitb14 \
    --lora_rank 16 \
    --lora_alpha 32 \
    --epochs 5 \
    --batch_size 16 \
    --lr 1e-4 \
    --curriculum_epochs 2 \
    --curriculum_start_frac 0.3 \
    --aw_min_radius 2 \
    --aw_max_radius 7 \
    --num_workers 4 \
    --output_dir ./checkpoints \
    --backup_dir /content/drive/MyDrive/AML/Checkpoints


In [ ]:
# ── Cella 6: Evaluation ─────────────────────────────────────────────────────
!python evaluate.py \
    --dataset_root ./data/SPair-71k \
    --checkpoint ./checkpoints/best.pth \
    --alpha 0.1

In [ ]:
# ── Cella 7: Salva checkpoint su Google Drive ───────────────────────────────
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
save_dir = "/content/drive/MyDrive/AML_Project5_checkpoints"
os.makedirs(save_dir, exist_ok=True)
shutil.copy("./checkpoints/best.pth", save_dir)
print(f"✅ Checkpoint salvato in {save_dir}")

## 6. (Opzionale) Ablation Study e Baseline
Qui puoi valutare la Baseline originale (DINOv2 puro senza addestramento) in un minuto e confrontarla coi tuoi pesi per dimostrare nella relazione quanto è migliorato!

In [ ]:
# Valutazione Baseline (DINOv2 pre-addestrato, NIENTE LoRA)
!python evaluate.py \
    --dataset_root ./data/SPair-71k \
    --baseline_only \
    --no_adaptive_win \
    --num_workers 4
